# Smart-Garden · Qwen3 Kaggle API

API dùng model `ToanPhamAIEngi/Qwen3-8B-Vietnam-MultiDomainEdu`, giữ đầy đủ system prompt/history và chỉ decode token mới sinh.

- Bật **Internet** và chọn GPU trong Kaggle Settings.
- Nếu dán các cell vào notebook đang có `model` và `tokenizer`, chúng được dùng lại; không tải model lần hai.
- Nếu mở notebook này độc lập, cell model sẽ nạp checkpoint một lần. Cần GPU/RAM phù hợp; có thể dùng cách nạp/quantization đang hoạt động của bạn thay cell này.
- Thêm secret **NGROK_AUTHTOKEN** trong Kaggle Secrets. **HF_TOKEN** và **CHAT_API_KEY** là tùy chọn.
- Mặc định dùng lại domain `sly-series-landowner.ngrok-free.dev`. Cell ngrok chỉ thay tunnel có đúng domain này. Đổi `NGROK_DOMAIN` nếu tài khoản của bạn dùng domain khác, hoặc đặt chuỗi rỗng để tự cấp URL.

Notebook không huấn luyện lại model. Đúng chat template không bảo đảm checkpoint luôn trả lời đúng; cần đọc kết quả kiểm tra cuối.


In [ ]:
%pip install -q fastapi==0.116.1 "uvicorn[standard]==0.35.0" pyngrok==9.1.0


In [ ]:
import os

def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

MODEL_ID = "ToanPhamAIEngi/Qwen3-8B-Vietnam-MultiDomainEdu"
if ("model" in globals()) != ("tokenizer" in globals()):
    raise RuntimeError("Cần cả model và tokenizer từ cùng checkpoint. Hãy hoàn tất cell nạp model đang dùng.")
if "model" not in globals():
    import torch
    import transformers
    from packaging.version import Version
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if not torch.cuda.is_available():
        raise RuntimeError("Chưa có GPU. Bật GPU trong Kaggle Settings rồi chạy lại.")
    if Version(transformers.__version__) < Version("4.51.0"):
        raise RuntimeError("Qwen3 cần transformers >= 4.51.0. Cập nhật thư viện rồi khởi động lại kernel trước khi nạp model.")
    token = read_secret("HF_TOKEN")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
    dtype_argument = {"dtype": "auto"} if Version(transformers.__version__) >= Version("4.56.0") else {"torch_dtype": "auto"}
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", token=token, **dtype_argument)
    print("Đã nạp model:", MODEL_ID)
else:
    print("Dùng lại model/tokenizer trong phiên hiện tại; không tải thêm weights.")


## Server API
Cell dưới được sinh từ `ai/kaggle_server.py`. Không thay system prompt của client bằng prompt cố định ở server.


In [ ]:
"""Kaggle inference API, reusing an already loaded Qwen3 model/tokenizer.

No model is downloaded on import. Notebook usage:
    engine = TransformersEngine(model, tokenizer, model_id=MODEL_ID)
    app = create_app(engine)
"""

import asyncio
from contextlib import asynccontextmanager
from dataclasses import dataclass
import hashlib
from importlib.metadata import PackageNotFoundError, version
import logging
import re
import secrets
from typing import Any, Literal, Protocol

from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, ConfigDict, Field, model_validator

SERVER_VERSION = "smart-garden-kaggle-v1"
DEFAULT_MODEL_ID = "ToanPhamAIEngi/Qwen3-8B-Vietnam-MultiDomainEdu"
DEFAULT_SYSTEM = (
    "Bạn là trợ lý nông nghiệp Smart-Garden. Tư vấn tiếng Việt dễ đọc, có căn cứ. "
    "Hỏi thêm khi thiếu loại cây, giai đoạn hoặc giá thể. Không tự tạo số đo, "
    "không khẳng định bệnh từ một triệu chứng và không có quyền điều khiển bơm. "
    "Chỉ xuất lời tư vấn cho người dùng. /no_think"
)
logger = logging.getLogger(__name__)


class Message(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1, max_length=64000)


class CompletionRequest(BaseModel):
    model_config = ConfigDict(extra="forbid", allow_inf_nan=False)
    messages: list[Message] = Field(min_length=1, max_length=32)
    temperature: float = Field(default=0.7, ge=0, le=2, strict=True)
    max_tokens: int = Field(default=1024, ge=1, le=2048, strict=True)

    @model_validator(mode="after")
    def validate_conversation(self) -> "CompletionRequest":
        turns = self.messages[1:] if self.messages[0].role == "system" else self.messages
        if not turns or len(turns) % 2 == 0 or any(
            turn.role != ("user" if index % 2 == 0 else "assistant")
            for index, turn in enumerate(turns)
        ):
            raise ValueError("Use an optional first system message, then alternating user/assistant turns ending in user")
        if sum(len(message.content) for message in self.messages) > 128000:
            raise ValueError("Conversation exceeds the input size limit")
        return self


@dataclass(frozen=True)
class GenerationResult:
    text: str
    input_tokens: int
    output_tokens: int


class ContextTooLong(ValueError):
    pass


class InferenceEngine(Protocol):
    model_id: str
    metadata: dict[str, Any]

    def generate(self, messages: list[dict[str, str]], temperature: float, max_tokens: int) -> GenerationResult: ...


def _package_version(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None


class TransformersEngine:
    def __init__(self, model: Any, tokenizer: Any, *, model_id: str = DEFAULT_MODEL_ID, max_context_tokens: int = 8192):
        if not isinstance(getattr(tokenizer, "chat_template", None), str) or not tokenizer.chat_template.strip():
            raise ValueError("Tokenizer của checkpoint chưa có chat_template; cần kiểm tra tokenizer trước khi mở API.")
        if max_context_tokens < 2049:
            raise ValueError("max_context_tokens must be at least 2049")
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.model_id = model_id
        model_limit = getattr(model.config, "max_position_embeddings", max_context_tokens)
        self.max_context_tokens = min(max_context_tokens, model_limit)
        self.metadata = {
            "checkpoint_revision": getattr(model.config, "_commit_hash", None),
            "loaded_model_name": getattr(model.config, "_name_or_path", None),
            "chat_template_sha256": hashlib.sha256(tokenizer.chat_template.encode()).hexdigest(),
            "transformers_version": _package_version("transformers"),
            "torch_version": _package_version("torch"),
            "max_context_tokens": self.max_context_tokens,
            "thinking_requested": False,
        }

    def generate(self, messages: list[dict[str, str]], temperature: float, max_tokens: int) -> GenerationResult:
        import torch

        # Preserve every role and turn; never replace the client system prompt
        # or collapse the input to messages[-1]['content'].
        inputs = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_tensors="pt",
            return_dict=True,
        ).to(self.model.device)
        input_length = inputs["input_ids"].shape[-1]
        if input_length + max_tokens > self.max_context_tokens:
            raise ContextTooLong("Conversation and requested output exceed the context budget")
        options = {
            "max_new_tokens": max_tokens,
            "do_sample": temperature > 0,
            "return_dict_in_generate": False,
            "output_scores": False,
            "pad_token_id": self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else self.tokenizer.eos_token_id,
        }
        if temperature > 0:
            options.update(temperature=temperature, top_p=0.8, top_k=20)
        with torch.inference_mode():
            generated = self.model.generate(**inputs, **options)
        # Decode generated tokens only; decoding the complete sequence leaks the
        # prompt/history and can look like the model is repeating instructions.
        new_tokens = generated[0][input_length:].tolist()
        text = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        return GenerationResult(text, input_length, len(new_tokens))


def final_text(text: str) -> str:
    text = re.sub(r"<think\b[^>]*>.*?(?:</think\s*>|$)", "", text, flags=re.I | re.S)
    return re.split(r"</think\s*>", text, flags=re.I)[-1].strip()


def create_app(engine: InferenceEngine, *, api_key: str | None = None) -> FastAPI:
    gate = asyncio.Lock()
    running: set[asyncio.Task] = set()

    @asynccontextmanager
    async def lifespan(_app: FastAPI):
        yield
        if running:
            await asyncio.gather(*running, return_exceptions=True)

    app = FastAPI(title="Smart-Garden Kaggle API", version=SERVER_VERSION, lifespan=lifespan)

    @app.get("/health")
    async def health() -> dict[str, Any]:
        return {
            "status": "ok", "model": engine.model_id, "server_version": SERVER_VERSION,
            "busy": gate.locked(), "inference": engine.metadata,
        }

    @app.post("/v1/chat/completions")
    async def complete(body: CompletionRequest, request: Request) -> dict[str, Any]:
        if api_key and not secrets.compare_digest(request.headers.get("Authorization", ""), f"Bearer {api_key}"):
            raise HTTPException(401, "invalid_api_key")
        if gate.locked():
            raise HTTPException(429, "model_busy", headers={"Retry-After": "5"})
        await gate.acquire()
        messages = [message.model_dump() for message in body.messages]
        if messages[0]["role"] != "system":
            messages.insert(0, {"role": "system", "content": DEFAULT_SYSTEM})
        task = asyncio.create_task(asyncio.to_thread(engine.generate, messages, body.temperature, body.max_tokens))
        running.add(task)

        def finished(job: asyncio.Task) -> None:
            running.discard(job)
            gate.release()
            # Retrieve failures even if the HTTP caller disconnected. The GPU
            # remains reserved until the real worker finishes, not until cancel.
            if not job.cancelled():
                job.exception()

        task.add_done_callback(finished)
        try:
            result = await asyncio.shield(task)
        except ContextTooLong:
            raise HTTPException(422, "context_too_long") from None
        except Exception as error:
            logger.warning("Generation failed: %s", type(error).__name__)
            raise HTTPException(503, "generation_failed") from None
        answer = final_text(result.text)
        if not answer:
            raise HTTPException(502, "empty_response")
        return {
            "response": answer,
            "model": engine.model_id,
            "server_version": SERVER_VERSION,
            "usage": {"prompt_tokens": result.input_tokens, "completion_tokens": result.output_tokens},
        }

    return app


In [ ]:
import json
import threading
import time
import urllib.request
import uvicorn

if "smart_garden_thread" in globals() and smart_garden_thread.is_alive():
    raise RuntimeError("Server này đã chạy. Dùng cell health, hoặc dừng server này trước khi khởi động lại.")
PORT = 8001  # Không giành cổng 8000 của server cũ.
MAX_CONTEXT_TOKENS = 8192
SMART_GARDEN_API_KEY = read_secret("CHAT_API_KEY")
engine = TransformersEngine(model, tokenizer, model_id=MODEL_ID, max_context_tokens=MAX_CONTEXT_TOKENS)
app = create_app(engine, api_key=SMART_GARDEN_API_KEY)
smart_garden_server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info"))
smart_garden_thread = threading.Thread(target=smart_garden_server.run, daemon=True)
smart_garden_thread.start()
for _ in range(80):
    if not smart_garden_thread.is_alive():
        raise RuntimeError("Server đã dừng lúc khởi động. Kiểm tra log/cổng 8001.")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1) as result:
            health = json.load(result)
        if health.get("server_version") == SERVER_VERSION:
            break
    except Exception:
        pass
    time.sleep(0.25)
else:
    raise RuntimeError("Server chưa sẵn sàng sau 20 giây.")
print(json.dumps(health, ensure_ascii=False, indent=2))


## Mở tunnel ngrok

Cell này thay tunnel có đúng domain Smart-Garden được cấu hình bên dưới sang server cổng 8001. Các tunnel domain khác được giữ nguyên. Không in token/key.


In [ ]:
from urllib.parse import urlparse
from pyngrok import ngrok

NGROK_DOMAIN = os.environ.get("NGROK_DOMAIN", "sly-series-landowner.ngrok-free.dev").strip()
ngrok_token = read_secret("NGROK_AUTHTOKEN")
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
if NGROK_DOMAIN:
    for existing in ngrok.get_tunnels():
        if urlparse(existing.public_url).hostname == NGROK_DOMAIN:
            ngrok.disconnect(existing.public_url)
tunnel_options = {"domain": NGROK_DOMAIN} if NGROK_DOMAIN else {}
smart_garden_tunnel = ngrok.connect(addr=PORT, proto="http", **tunnel_options)
API_URL = smart_garden_tunnel.public_url.rstrip("/") + "/v1/chat/completions"
HEALTH_URL = smart_garden_tunnel.public_url.rstrip("/") + "/health"
print("API_URL:", API_URL)
print("Health:", HEALTH_URL)
print("Nếu URL đổi, cập nhật CHAT_API_URL trong .env của Smart-Garden.")


## Kiểm tra system prompt và lịch sử

Phép thử này kiểm tra checkpoint có làm theo một chỉ dẫn đơn giản và nhớ thông tin ở lượt trước hay không. Kết quả không đạt cần được xem lại trước khi đánh giá nông học; không coi HTTP 200 là model trả lời đúng.


In [ ]:
def local_chat(messages, max_tokens=128):
    body = json.dumps({"messages": messages, "temperature": 0.0, "max_tokens": max_tokens}, ensure_ascii=False).encode("utf-8")
    headers = {"Content-Type": "application/json"}
    if SMART_GARDEN_API_KEY:
        headers["Authorization"] = "Bearer " + SMART_GARDEN_API_KEY
    request = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", data=body, headers=headers, method="POST")
    with urllib.request.urlopen(request, timeout=180) as response:
        return json.load(response)

marker = "SMART_GARDEN_SYSTEM_OK"
system_probe = local_chat([
    {"role": "system", "content": f"Chỉ trả lời chính xác chuỗi {marker}. Không thêm bất kỳ nội dung nào."},
    {"role": "user", "content": "Xin chào."},
])
history_probe = local_chat([
    {"role": "system", "content": "Trả lời ngắn bằng tiếng Việt, chỉ dựa trên thông tin đã nêu."},
    {"role": "user", "content": "Tôi trồng cà chua trong chậu, không phải rau muống."},
    {"role": "assistant", "content": "Mình đã ghi nhận loại cây của bạn."},
    {"role": "user", "content": "Tôi đang trồng cây gì?"},
])
print(json.dumps({
    "system_instruction_ok": system_probe["response"].strip() == marker,
    "system_probe": system_probe,
    "history_probe": history_probe,
}, ensure_ascii=False, indent=2))


## Kết nối về dashboard

1. Giữ phiên Kaggle đang chạy. Trong `.env` ở máy tính, đặt `CHAT_API_URL` bằng URL in ở trên. Nếu đặt secret `CHAT_API_KEY`, điền cùng key trong `.env`; không gửi key vào chat.
2. Chạy `docker compose up -d --force-recreate backend` tại thư mục Smart-Garden.
3. Kiểm tra health public có `server_version: smart-garden-kaggle-v1`, rồi mở dashboard để thử lại.

Muốn dừng **server này**, chạy riêng một cell:

```python
ngrok.disconnect(smart_garden_tunnel.public_url)
smart_garden_server.should_exit = True
smart_garden_thread.join(timeout=20)
```

Không dùng `ngrok.kill()` hoặc xóa model nếu phiên notebook còn tác vụ khác. Server bận trả 429; không tự sinh câu trả lời giả khi GPU lỗi hoặc hết context. Không tự cắt system/history khi vượt giới hạn token.
